# P2.3 · MLP regression on California housing (MSE vs Huber, RMSE and R² by hand)

**ID:** P2.3 · **Target:** 25 min · **Plan week:** 1 (timed repeat in week 3) · **Part:** P (PyTorch first principles)

**The drill.** Predict median house value (in units of \$100k) for 20 640 California block groups from 8 numeric
features. Use the P0 skeleton, standardize **X and y with train statistics only**, train the same MLP with
`MSELoss` and with `HuberLoss`, and report RMSE and R² **in original units**, computed by hand and checked
against scikit-learn.

**Must include (from the plan):** standardization without leakage, MSE vs Huber, RMSE and R². Also: trivial
baselines (predict the train mean; linear regression), initial-loss check (≈ 1 for standardized targets),
overfit-one-batch, `(N,)` vs `(N, 1)` shape discipline, eval over the whole set under `eval()` + `no_grad()`.

**Rules.** No docs, no LLM. The data is cached: `fetch_california_housing(data_home=DATA/"sklearn",
download_if_missing=False)`. Each solution cell keeps only its spec. The self-check names the variables it expects.

## Sources
- R. K. Pace, R. Barry, *Sparse spatial autoregressions*, Statistics & Probability Letters 33(3):291-297, 1997,
  DOI 10.1016/S0167-7152(96)00140-X — https://doi.org/10.1016/S0167-7152(96)00140-X (origin of the dataset).
- scikit-learn *California Housing dataset* description (features, target in \$100k, block groups):
  https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset
- scikit-learn `r2_score` / `root_mean_squared_error` (references for the by-hand checks):
  https://scikit-learn.org/stable/modules/model_evaluation.html#r2-score
- P. J. Huber, *Robust Estimation of a Location Parameter*, Ann. Math. Statist. 35(1):73-101, 1964,
  DOI 10.1214/aoms/1177703732 — https://doi.org/10.1214/aoms/1177703732
- `torch.nn.HuberLoss` docs (δ, relation to `SmoothL1Loss`): https://docs.pytorch.org/docs/stable/generated/torch.nn.HuberLoss.html
- `torch.nn.MSELoss` docs (broadcasting warning on mismatched shapes):
  https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html
- scikit-learn user guide, *Common pitfalls: data leakage during pre-processing*:
  https://scikit-learn.org/stable/common_pitfalls.html#data-leakage

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from prep import DATA, Session, check, check_close, seed_everything

s = Session("P2.3", target_min=25)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. DATA. fetch_california_housing(data_home=DATA/"sklearn", download_if_missing=False) -> X (20640, 8), y (20640,).
#    Inspect feature names, ranges, the target's max (it is CAPPED: how many rows sit at the cap?).
#    Split 70/15/15 with train_test_split(test_size=0.30, random_state=0) then (test_size=0.50, random_state=0)
#    -> numpy X_train_np, X_val_np, X_test_np, y_train_np, y_val_np, y_test_np.
#    Standardize X with train mean/std -> `x_mu`, `x_sd` and y with train mean/std -> `y_mu`, `y_sd` (floats).
#    Tensors: X_train, X_val, X_test (float32, (N, 8)); y_train, y_val, y_test standardized float32 of shape (N,).
# YOUR CODE HERE


In [ ]:
# 2. METRICS BY HAND (numpy or torch, ORIGINAL units).
#    rmse(y, yhat) = sqrt(mean((y - yhat)^2)).
#    r2(y, yhat) = 1 - SS_res / SS_tot, SS_tot = sum((y - mean(y))^2) using the mean of the TRUE y.
#    Baselines on val (original units): predict the train mean (`base_mean` dict with "rmse", "r2"),
#    and sklearn LinearRegression on the standardized X (`base_lin` dict).
# YOUR CODE HERE


**The `(N, 1)` vs `(N,)` trap.** Run this before you write the model: the "loss" below is not the MSE of the
predictions, even though it returns a number.

In [ ]:
pred_col = (0.8 * y_val + 0.1)[:, None]  # (N, 1): a decent predictor whose Linear(…, 1) output you forgot to squeeze
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    bad = nn.functional.mse_loss(pred_col, y_val)  # broadcasts to (N, N)!
print(f"broadcast 'MSE' {bad:.4f} vs correct {nn.functional.mse_loss(pred_col.squeeze(1), y_val):.4f};",
      "warning:", w[0].category.__name__ if w else None)
print("hand-written version broadcasts silently:", ((pred_col - y_val) ** 2).shape)

In [ ]:
# 3. MODEL. `MLP`: 8 -> 128 -> 128 -> 1, ReLU, forward returns shape (B,) (squeeze the last dim).
#    `make_model()` -> fresh model on `dev`. DataLoaders: train batch 256 shuffle; val/test batch 4096.
#    `predict(model, X) -> ORIGINAL-unit numpy predictions` (model.eval(), no_grad, then * y_sd + y_mu).
# YOUR CODE HERE


In [ ]:
# 4. SANITY CHECKS. (a) `init_mse`: MSE of a fresh model on standardized val targets (≈ Var(y_std) ≈ 1).
#    (b) Overfit one batch of 64 with a fresh model, Adam lr=1e-3, 500 steps, MSE -> `overfit_loss` (< 0.01).
# YOUR CODE HERE


In [ ]:
# 5. TRAIN FUNCTION. `fit(loss_fn, epochs=40, lr=1e-3) -> (model, history)` on a FRESH seeded model
#    (torch.manual_seed(0) inside, so MSE and Huber start from the same init): AdamW (wd=1e-4),
#    CosineAnnealingLR over the epochs, per epoch: train with `loss_fn` on standardized y; record train loss and
#    val RMSE in ORIGINAL units (via predict) in history["train_loss"], history["val_rmse"].
#    Keep the best-val-RMSE state_dict and load it before returning.
# YOUR CODE HERE


In [ ]:
# 6. MSE vs HUBER. results = {"mse": ..., "huber": ...}, each a dict with "model", "history",
#    "val_rmse", "val_mae", "val_r2" (ORIGINAL units, by hand; MAE = mean |y - yhat|).
#    Huber: nn.HuberLoss(delta=1.0) on standardized targets (delta is in units of y_sd: say what that means).
#    Plot val RMSE curves of both, and residuals (yhat - y) vs y on val for the MSE model: look at the cap.
# YOUR CODE HERE


In [ ]:
# 7. TEST (once). Pick the loss with the lower val RMSE -> `best_name`. On test, ORIGINAL units:
#    `test_rmse`, `test_r2`, `test_mae`. Print RMSE also in dollars.
# YOUR CODE HERE


## Self-check
Expects: `x_mu`, `x_sd`, `y_mu`, `y_sd`, `X_train`, `y_train`, `rmse`, `r2`, `base_mean`, `init_mse`, `overfit_loss`,
`results` (keys "mse", "huber" each with "model", "val_rmse", "val_r2"), `predict`, `test_rmse`, `test_r2`.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

_X, _y = fetch_california_housing(data_home=DATA / "sklearn", download_if_missing=False, return_X_y=True)
_Xtr, _, _ytr, _ = train_test_split(_X, _y, test_size=0.30, random_state=0)
check_close("x_mu from TRAIN split only", np.asarray(x_mu, dtype=np.float64), _Xtr.mean(0), rtol=1e-6)
check_close("x_sd from TRAIN split only", np.asarray(x_sd, dtype=np.float64), _Xtr.std(0), rtol=1e-3)
check_close("y_mu / y_sd from TRAIN split only", [y_mu, y_sd], [_ytr.mean(), _ytr.std()], rtol=1e-3)
check("targets are (N,) float", y_train.ndim == 1 and y_train.dtype == torch.float32, f"got {tuple(y_train.shape)} {y_train.dtype}")
rng = np.random.default_rng(0)
a, b = rng.normal(2, 1, 300), rng.normal(2, 1.5, 300)
check_close("rmse == sklearn", rmse(a, b), math.sqrt(mean_squared_error(a, b)))
check_close("r2 == sklearn", r2(a, b), r2_score(a, b))
check_close("r2(y, y) == 1", r2(a, a), 1.0)
check_close("r2 of the mean predictor == 0", r2(a, np.full_like(a, a.mean())), 0.0, atol=1e-9)
check("train-mean baseline R2 ~ 0 on val", abs(base_mean["r2"]) < 0.01, f"got {base_mean['r2']:.4f}")
check("model output is (B,)", results["mse"]["model"](X_val[:5].to(dev)).shape == (5,))
check("initial MSE ~ 1 (standardized y)", 0.5 < init_mse < 2.0, f"got {init_mse:.3f}")
check("can overfit one batch", overfit_loss < 0.01, f"got {overfit_loss:.5f}")
for k in ("mse", "huber"):
    pk = predict(results[k]["model"], X_val)
    check(f"{k}: predict() returns original units", abs(pk.mean() - y_val_np.mean()) < 0.3, f"mean pred {pk.mean():.3f}")
    check_close(f"{k}: stored val R2 matches sklearn", results[k]["val_r2"], r2_score(y_val_np, pk), atol=1e-6)
    check(f"{k}: beats linear regression", results[k]["val_rmse"] < base_lin["rmse"] - 0.05,
          f"{results[k]['val_rmse']:.3f} vs linear {base_lin['rmse']:.3f}")
check("test R2 > 0.75", test_r2 > 0.75, f"got {test_r2:.3f}")
check("test RMSE < 0.60 ($100k)", test_rmse < 0.60, f"got {test_rmse:.3f}")

## Say it out loud (P3)
1. Why standardize y as well as X? What goes wrong if you don't (think: init, LR, loss scale)?
2. Which likelihood does MSE maximize? Which does L1? Where does Huber sit, and what does δ = 1 mean here?
3. Why did the initial MSE come out ≈ 1? What would it be if you had *not* standardized y?
4. RMSE vs MAE vs R²: what does each tell a stakeholder? Can R² be negative?
5. The target is capped at 5.00001 (\$500k). How does that show up in the residual plot, and how would you handle it?
6. Where exactly can leakage enter this pipeline (list three places)?
7. `(N, 1)` vs `(N,)`: which PyTorch calls broadcast, which raise, which only warn?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up